# Phase B Session 4: Top-3 Architecture Validation
## Validate loss ranking from Session 3 with longer training

**Session 3 Top 3 by Loss:**
1. W64-D3-NoSkip (loss=0.897)
2. W64-D3-Skip (loss=0.912)
3. W64-D5-Skip (loss=0.912)

**Goal**: Verify if 5-epoch ranking holds at 50 epochs

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Done")

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/src')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import json, os

print("Imports OK")

In [ ]:
# Load Session 2 calibration indices (exclude them)
s2_path = '/kaggle/working/ThermoRG-NN/experiments/phase_b/phase_b_session2_results.json'
if os.path.exists(s2_path):
    with open(s2_path) as f:
        s2 = json.load(f)
    calib_idx = set(s2['calib_indices'])
    print(f"Excluded calibration indices: {len(calib_idx)}")
else:
    calib_idx = set(range(5000))
    print("Using fallback: exclude 0-4999")

In [ ]:
# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

all_idx = set(range(len(full_train)))
main_idx = list(all_idx - calib_idx)
main_set = Subset(full_train, main_idx)

print(f"Training set: {len(main_set)} images")

batch_size = 128
main_loader = DataLoader(main_set, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
class Net(nn.Module):
    def __init__(self, w=32, d=3, skip=False, norm='none'):
        super().__init__()
        self.skip = skip
        self.conv1 = nn.Conv2d(3, w, 3, padding=1, bias=False)
        self.bn1 = self._norm(w, norm)
        self.conv2 = nn.Conv2d(w, w, 3, padding=1, bias=False)
        self.bn2 = self._norm(w, norm)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Linear(w * 16, 10)
        self.act = nn.GELU()
        if skip:
            self.skip_conv = nn.Conv2d(w, w, 1, bias=False)
    
    def _norm(self, c, t):
        if t == 'bn': return nn.BatchNorm2d(c)
        if t == 'ln': return nn.LayerNorm([c, 32, 32])
        return nn.Identity()
    
    def forward(self, x):
        x = self.act(self.bn1(self.conv1(x)))
        s = x if not self.skip else self.skip_conv(x)
        x = self.act(self.bn2(self.conv2(x)))
        if self.skip: x = x + s
        return self.fc(self.pool(x).view(x.size(0), -1))

print("Net defined")

In [ ]:
def train_epoch(model, loader, opt, crit, dev):
    model.train()
    total = 0
    for X, y in loader:
        X, y = X.to(dev), y.to(dev)
        opt.zero_grad()
        loss = crit(model(X), y)
        loss.backward()
        opt.step()
        total += loss.item()
    return total / len(loader)

def evaluate(model, loader, dev):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(dev), y.to(dev)
            pred = model(X).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Session 3 Top 3 by loss
candidates = [
    {"name": "W64-D3-NoSkip", "w": 64, "d": 3, "skip": False, "norm": "bn"},
    {"name": "W64-D3-Skip",   "w": 64, "d": 3, "skip": True,  "norm": "bn"},
    {"name": "W64-D5-Skip",   "w": 64, "d": 5, "skip": True,  "norm": "bn"},
]

EPOCHS = 50
print(f"Evaluating {len(candidates)} candidates x {EPOCHS} epochs")

In [ ]:
# Train and evaluate
records = []

for i, c in enumerate(candidates):
    print(f"\n[{i+1}/{len(candidates)}] {c['name']}...")
    
    m = Net(w=c['w'], d=c['d'], skip=c['skip'], norm=c['norm']).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    
    loss_history = []
    for ep in range(EPOCHS):
        l = train_epoch(m, main_loader, opt, crit, device)
        loss_history.append(l)
        if (ep + 1) % 10 == 0:
            acc = evaluate(m, test_loader, device)
            print(f"  ep{ep+1}: loss={l:.4f}, acc={acc:.4f}")
    
    final_acc = evaluate(m, test_loader, device)
    
    records.append({
        "arch": c['name'],
        "final_loss": loss_history[-1],
        "final_acc": final_acc,
        "loss_history": loss_history,
        "session3_loss5": c.get('loss5', 'N/A'),
    })
    print(f"  Final: loss={loss_history[-1]:.4f}, acc={final_acc:.4f}")

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"arch": r['arch'], "loss5": r['session3_loss5'], "loss50": r['final_loss'], "acc50": r['final_acc']}
    for r in records
])
df['rank_session3'] = df['loss5'].rank(ascending=True).astype(int)
df['rank_session4'] = df['loss50'].rank(ascending=True).astype(int)
df = df.sort_values('rank_session4')

print("Session 3 vs Session 4 Ranking:")
print(df.to_string(index=False))

print(f"\n{'='*60}")
print("SESSION 4 RESULTS:")
for _, row in df.iterrows():
    print(f"  {row['arch']}: acc={row['acc50']:.4f} (S3 rank={row['rank_session3']}, S4 rank={row['rank_session4']})")
print(f"{'='*60}")

In [ ]:
from datetime import datetime

out = {
    "date": datetime.now().isoformat(),
    "session": "Session 4 (50 epochs)",
    "epochs": EPOCHS,
    "calib_excluded": len(calib_idx),
    "records": [
        {"arch": r['arch'], "loss5": r['session3_loss5'], "loss50": r['final_loss'], "acc50": r['final_acc']}
        for r in records
    ],
}

out_path = '/kaggle/working/phase_b_session4_results.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f"Saved to {out_path}")